# DSE Baselines v2 — Notebook 1: Pure PyTorch (`MSSF_v2`, `SDPred`, `MGPred`)

Chạy lại 3 baseline PyTorch thuần dưới **Protocol v2 thống nhất** (`load_splits_v2`, không dùng `sed`, không rò rỉ dữ liệu, chọn checkpoint chính theo `MacroF1` trên `val_select` + lưu kèm checkpoint phụ theo `RMSE` cho `SDPred`/`MGPred` tại `*_rmseSel_*`).

In [ ]:
# ===== 1. CLONE / UPDATE REPO & MOUNT GOOGLE DRIVE =====
import os, subprocess, torch

REPO_URL = 'https://github.com/hungbuile04/DSE.git'
REPO_BRANCH = 'main'
WORK_DIR = '/content/DSE'

if not os.path.exists(WORK_DIR):
    subprocess.run(['git', 'clone', '-b', REPO_BRANCH, REPO_URL, WORK_DIR], check=True)
else:
    subprocess.run(['git', '-C', WORK_DIR, 'pull'], check=False)

os.chdir(WORK_DIR)
print('Working directory:', os.getcwd())

# Mount Google Drive for per-fold auto-backup & seamless resume
DRIVE_DIR = '/content/drive/MyDrive/DSE_results_v2'
try:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(DRIVE_DIR, exist_ok=True)
    print(f'Google Drive mounted. Auto-sync directory: {DRIVE_DIR}')
except Exception as e:
    DRIVE_DIR = '/content/DSE_drive_fallback'
    os.makedirs(DRIVE_DIR, exist_ok=True)
    print(f'Google Drive not mounted ({e}); using local fallback: {DRIVE_DIR}')

os.environ['DRIVE_DIR'] = DRIVE_DIR

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    try:
        torch.set_float32_matmul_precision('medium')
    except Exception:
        pass
    print(f'GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB)')
else:
    print('WARNING: CUDA GPU not detected! Enable Runtime -> Change runtime type -> GPU.')

!pip install -q numpy scipy pandas scikit-learn

In [ ]:
# ===== 2. VERIFY BENCHMARK DATA INTEGRITY (SHA256) =====
!python MSSF_plus/audit/verify_checksums.py --check

---
## 3. Baseline 1: `MSSF_v2` (Split B & Split A — 10 Folds)

In [ ]:
%%time
!python -m MSSF_plus.baselines_rerun.run_all \
    --baselines MSSF_v2 \
    --splits B A \
    --mode full \
    --results_dir ./results_v2 \
    --drive_dir "$DRIVE_DIR" \
    --resume

---
## 4. Baseline 2: `SDPred` (Split B & Split A — 10 Folds)

In [ ]:
%%time
!python -m MSSF_plus.baselines_rerun.run_all \
    --baselines SDPred \
    --splits B A \
    --mode full \
    --results_dir ./results_v2 \
    --drive_dir "$DRIVE_DIR" \
    --resume

---
## 5. Baseline 3: `MGPred` (Split B & Split A — 10 Folds)
Đã sửa lỗi zero đặc trưng tĩnh STITCH/Mol2vec và lỗi `ValueError: inhomogeneous shape` cho thuốc cold-start ở Split B.

In [ ]:
%%time
!python -m MSSF_plus.baselines_rerun.run_all \
    --baselines MGPred \
    --splits B A \
    --mode full \
    --results_dir ./results_v2 \
    --drive_dir "$DRIVE_DIR" \
    --resume

---
## 6. Kiểm tra kết quả đã lưu trên Google Drive

In [ ]:
!python -m MSSF_plus.scripts.summarize_results --results_dir "$DRIVE_DIR" --ref_method MSSF_v2
!python shared_data/unified_eval.py --results_dir "$DRIVE_DIR" 